In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Paths -- edit these for your environment.
RAW_DIR  = "/PATH/TO/raw"              # raw MGB extracts (edw/, clarity/, Datamart/)
DATA_DIR = "/PATH/TO/data"             # project data root; lookup tables live here
WORK_DIR = f"{DATA_DIR}/20250604"      # identified intermediate outputs -- keep on the secure server
DEID_DIR = f"{WORK_DIR}/deidentified"  # de-identified outputs and ID / date-shift maps

In [ ]:
## Load in cultures cohort
microbiology_cultures = pd.read_csv(f'{WORK_DIR}/microbiology_cultures_cohort.csv')
demo = pd.read_csv(f'{RAW_DIR}/edw/demographics/MGB_demographics_20240715.csv')

In [ ]:
demo = demo.rename(columns = {'PatientID':'anon_id',
                              'DOB':'birth_date_jittered',
                              'sex':'gender'})

In [ ]:
demo_final = microbiology_cultures.merge(demo,
                                         on = ['anon_id'])
demo_final['age'] = round((pd.to_datetime(demo_final['order_time_jittered_utc']).dt.tz_localize(tz=None) - pd.to_datetime(demo_final['birth_date_jittered'])).dt.days / 365.25,2)
demo_final = demo_final[['anon_id','pat_enc_csn_id_coded','order_proc_id_coded','age','gender']].copy()
demo_final = demo_final.drop_duplicates()
demo_final

In [ ]:
## Clean-up Gender
demo_final['gender'] = demo_final['gender'].apply(lambda x: x if x in ['Male', 'Female'] else 'Others')
demo_final

In [ ]:
demo_final.to_csv(f'{WORK_DIR}/microbiology_cultures_demographics.csv',
                  index = False)

## Deidentification

In [ ]:
# Load the saved mappings (assume you saved as CSVs before)
anon_map_df = pd.read_csv(f'{DEID_DIR}/anon_id_map.csv')
enc_map_df = pd.read_csv(f'{DEID_DIR}/enc_id_map.csv')
order_proc_map_df = pd.read_csv(f'{DEID_DIR}/order_proc_id_map.csv')

In [ ]:
# Convert to Dict
anon_id_map = dict(zip(anon_map_df['anon_id'], anon_map_df['anon_id_deid']))
enc_id_map = dict(zip(enc_map_df['pat_enc_csn_id_coded'], enc_map_df['pat_enc_csn_id_deid']))
order_proc_map = dict(zip(order_proc_map_df['order_proc_id_coded'], order_proc_map_df['order_proc_id_deid']))

# Apply the mappings
demo_deid = demo_final.copy()
demo_deid['anon_id'] = demo_deid['anon_id'].map(anon_id_map)
demo_deid['pat_enc_csn_id_coded'] = demo_deid['pat_enc_csn_id_coded'].map(enc_id_map)
demo_deid['order_proc_id_coded'] = demo_deid['order_proc_id_coded'].map(order_proc_map)

# Bin the age column into categories
age_bins = [0, 24, 34, 44, 54, 64, 74, 84, 89, float('inf')]
age_labels = ['18-24 years', '25-34 years', '35-44 years', '45-54 years',
              '55-64 years', '65-74 years', '75-84 years', '85-89 years', 'above 90']

demo_deid['age'] = pd.cut(demo_deid['age'], bins=age_bins, labels=age_labels, right=True)
demo_deid

In [ ]:
demo_deid.to_csv(f'{DEID_DIR}/microbiology_cultures_demographics.csv',
                 index = False)